# Figure 08 -- strong scaling

Loads `bench/results/multigpu/strong_scaling.json`, produced by `bench/multigpu/strong_scaling.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("multigpu/strong_scaling.json")
cfg, recs = art["config"], art["data"]["records"]

# A point whose traversal buffers overflowed produced a TRUNCATED force, so its
# wall clock is padded-buffer overhead over a wrong answer. Those are dropped
# from the curve and reported on the figure rather than silently omitted -- the
# regime boundary is the result here, not an inconvenience.
ok = [r for r in recs if r.get("valid")]
bad = [r for r in recs if not r.get("valid")]
if not ok:
    raise SystemExit(
        "strong_scaling.json has no valid points: every device count overflowed. "
        "Lower the fixed N, or raise leaf_size and the traversal caps."
    )
ok.sort(key=lambda r: r["ndev"])

ndev = [r["ndev"] for r in ok]
t = [r["median_s"] for r in ok]
# Efficiency is referenced to the smallest VALID device count, not to 1: a
# single-device run is not the same code path.
ref_d, ref_t = ndev[0], t[0]
eff = [(ref_t * ref_d) / (ti * di) for di, ti in zip(ndev, t)]

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2)

axes[0].plot(ndev, t, marker=style.MARKERS[0], color=style.entity_color("wall", 0))
axes[0].set_xlabel("devices")
axes[0].set_ylabel("time per force evaluation [s]")
axes[0].set_yscale("log")
axes[0].set_title("wall clock at fixed $N$", fontsize=8)
style.finish(axes[0], legend=False)

axes[1].plot(ndev, eff, marker=style.MARKERS[1], color=style.entity_color("eff", 1),
             label="measured")
axes[1].axhline(1.0, linestyle=":", linewidth=0.8, color="0.5", label="ideal")
axes[1].set_xlabel("devices")
axes[1].set_ylabel(f"parallel efficiency (ref. {ref_d} devices)")
axes[1].set_ylim(0, 1.15)
axes[1].set_title("efficiency", fontsize=8)
style.finish(axes[1], legend=True, legend_kwargs={"loc": "best", "fontsize": 6.2})

note = jsonio.config_caption(cfg, ["order", "leaf_size", "precision", "device"])
if bad:
    dropped = ", ".join(f"{r['ndev']}x" for r in sorted(bad, key=lambda r: r["ndev"]))
    note += (
        f"\ndropped (buffer overflow, force truncated): {dropped}"
        f"  |  healthy load {cfg['healthy_per_device_n']}/device"
    )
style.annotate_config(axes[0], note)
style.save(fig, FIG_DIR / "fig08_strong_scaling.pdf")

for r in sorted(recs, key=lambda r: r["ndev"]):
    flag = "" if r.get("valid") else "   INVALID (overflow: %s)" % ",".join(r.get("overflowed", []))
    print(f"ndev={r['ndev']:<3d} n={r.get('n')} per_dev={r.get('per_device_n')} "
          f"median={r.get('median_s')}{flag}")


## Caption

Strong scaling at fixed total $N$: time per force evaluation and parallel
efficiency against device count. Efficiency is referenced to the smallest device
count that produced a valid force, not to a single device, because the
single-device path is a different code path. Points at which a traversal buffer
overflowed are excluded and named on the figure: an overflow truncates the force,
so those runs are not slow correct answers but fast wrong ones. The excluded
region is the low-device-count end, where the per-device load is highest.